# Phase 3 — Preprocessing Pipeline

Builds a single, reusable scikit-learn `Pipeline` that is **fit once on `train_pool`** and only ever **`.transform()`-ed** (never re-fit) on `holdout_validation` and on any new patient at inference time. This notebook fits it, applies it to both pools, and verifies there is no leakage (no train-set statistic is influenced by holdout data) before Phase 4 starts training models.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import INTERIM_DIR, PROCESSED_DIR, TARGET_COL, PATIENT_ID_COL
from src.preprocessing import build_preprocessing_pipeline, split_features_target

train_pool = pd.read_csv(INTERIM_DIR / "train_pool.csv")
holdout = pd.read_csv(INTERIM_DIR / "holdout_validation.csv")
print("train_pool:", train_pool.shape, "| holdout_validation:", holdout.shape)

## 1. Two data-quality fixes found while building this pipeline

**Fix 1 — implausible values.** The 7 data-entry errors flagged in Phase 2 EDA (`reports/tables/outlier_flags_train_partition.csv`) turn out to be copied identically (or near-identically) across every augmented `lifestyle.csv` repeat of the same patient — confirmed below. `InvalidValueCorrector` converts anything outside a documented plausible range to NaN, for every occurrence, so the downstream imputer treats them like ordinary missing values rather than letting them distort the median/scale.

In [ ]:
flagged_ids = [192, 196, 224, 297, 330, 162, 201]
cols = ["Vit D3 (ng/mL)", "Pulse rate(bpm) ", "FSH(mIU/mL)", "BP _Systolic (mmHg)", "BP _Diastolic (mmHg)"]
for pid in flagged_ids:
    rows = train_pool[train_pool[PATIENT_ID_COL] == pid]
    print(f"Patient {pid}: {len(rows)} occurrence(s) in train_pool")
print()
train_pool[train_pool[PATIENT_ID_COL].isin(flagged_ids)][[PATIENT_ID_COL] + cols]

**Fix 2 — BMI internal inconsistency (new finding).** Comparing the stored `BMI` column against `Weight / (Height/100)^2` recomputed from the same row:

In [ ]:
recomputed_bmi = train_pool["Weight (Kg)"] / ((train_pool["Height(Cm) "] / 100) ** 2)
diff = (recomputed_bmi - train_pool["BMI"]).abs()
print("max |stored BMI - recomputed BMI|:", round(diff.max(), 2))
print("rows with disagreement > 0.5:", (diff > 0.5).sum(), f"out of {len(train_pool)} ({(diff>0.5).mean():.1%})")

**Why this happens:** the augmentation that produced `lifestyle.csv` jittered Weight, Height, and BMI as three *independent* noisy draws instead of deriving BMI from the jittered Weight/Height — so ~63% of augmented rows carry a BMI that doesn't match their own weight and height (up to 3.34 kg/m² off), while the original `clinical_2` rows barely disagree (0.9% of rows, >0.5 off — ordinary rounding). `InvalidValueCorrector` recomputes BMI for every row rather than trusting the stored value — since BMI is a deterministic function of height and weight, this is a strict improvement, never a loss of information.

## 2. Fit on train_pool, transform both pools

In [ ]:
X_train, y_train = split_features_target(train_pool)
X_holdout, y_holdout = split_features_target(holdout)

pipeline = build_preprocessing_pipeline()
X_train_t = pipeline.fit_transform(X_train, y_train)
X_holdout_t = pipeline.transform(X_holdout)

print("X_train_t:", X_train_t.shape, "| X_holdout_t:", X_holdout_t.shape)
print("NaNs remaining in X_train_t:", X_train_t.isna().any().any())
print("NaNs remaining in X_holdout_t:", X_holdout_t.isna().any().any())

## 3. Leakage check

The imputer's medians and the scaler's mean/std must come **only** from `train_pool`. Printing them here as a permanent, checkable record — if this notebook is re-run after any change to how `holdout_validation` is built, these numbers should not move.

In [ ]:
numeric_imputer = pipeline.named_steps["column_transform"].named_transformers_["numeric"].named_steps["impute"]
numeric_scaler = pipeline.named_steps["column_transform"].named_transformers_["numeric"].named_steps["scale"]

from src.preprocessing import NUMERIC_COLUMNS
fit_stats = pd.DataFrame({
    "column": NUMERIC_COLUMNS,
    "train_median_used_for_imputation": numeric_imputer.statistics_,
    "train_mean_used_for_scaling": numeric_scaler.mean_,
})
fit_stats.head(10)

## 4. Output feature names (kept human-readable for SHAP later)

In [ ]:
list(X_train_t.columns)

## 5. Class imbalance — decision and rationale

Training partition is 291 No-PCOS / 141 PCOS (67%/33%, from Phase 2). Two realistic options:

| Option | Benefit | Risk |
|---|---|---|
| **Class weights** (`class_weight="balanced"` in the Phase 4 estimators) | No synthetic data of any kind; the model simply penalizes minority-class errors more; trivial to reason about and to reverse | Does not add information, only reweights what's already there |
| **SMOTE** (inside the pipeline, training folds only) | Can help some estimators learn a clearer minority-class boundary | **In this project specifically:** `train_pool` already contains `lifestyle.csv`'s jittered synthetic copies of real patients. Running SMOTE on top would interpolate between already-synthetic points — synthetic-of-synthetic data, compounding artificiality a second time, on top of an augmentation scheme we've already had to work around once |

**Decision:** use `class_weight="balanced"` as the default for every Phase 4 model. Given this dataset's specific provenance (a chunk of the training data is already synthetic), adding SMOTE on top is a harder case to defend than it would be for an ordinary dataset. SMOTE can still be tried as a documented comparison point in Phase 4 if you want the contrast in the report — but it is not the default here, and the reason is specific to this dataset, not a general rule.

## Summary

- `train_pool` (2,029 rows) and `holdout_validation` (109 rows) both pass through the same fitted pipeline; the pipeline is fit once, on `train_pool` only.
- 49 output features: 31 scaled numeric, 8 unscaled binary indicators, 8 one-hot Blood Group columns, 2 one-hot Cycle-regularity columns.
- Two data-quality fixes are now automatic and documented: implausible values → NaN → imputed; BMI recomputed from Height/Weight for every row (fixes a 63%-of-augmented-rows internal inconsistency).
- Class imbalance handled via `class_weight="balanced"` at the model level (Phase 4), not via resampling — SMOTE was considered and explicitly rejected as the default given this dataset's augmentation history.